
# Build Pathway–Gene Topology

Construct a pathway-informed gene interaction topology from Reactome pathway relationships and NCBI gene mappings.

## Workflow

1. Load Reactome pathway graph
2. Load Reactome–NCBI mappings
3. Load approved gene symbols
4. Filter to valid genes
5. Build pathway–gene associations
6. Generate pathway-pair → gene-pair relationships
7. Remove duplicates
8. Limit graph density
9. Export pathway-informed topology


In [1]:
import pandas as pd
import numpy as np
from itertools import product
from tqdm.auto import tqdm
import random
import os

tqdm.pandas()

# ==============================
# CONFIG
# ==============================

MAX_GENES_PER_PATHWAY = 500
RANDOM_TRUNCATION = False
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


# ==============================
# 0. Load GENE LABEL UNIVERSE
# ==============================

LABEL_FILE = (
    "data/processed/gene_labels_driver_vs_nondriver.csv"
)

gene_labels = pd.read_csv(LABEL_FILE)

# Use the gene column from the label file
# Adjust "gene" below if the actual column name differs.
omics_genes = set(
    gene_labels["gene"]
    .astype(str)
    .str.upper()
    .str.strip()
)

print(
    f"✅ Gene universe from driver/non-driver labels: "
    f"{len(omics_genes)} genes"
)


# ==============================
# 1. Load driver / non-driver
# ==============================

drivers = pd.read_table(
    "data/processed/driver_genes.txt",
    # "../data/processed/763_driver_genes.txt",
    header=None
)[0].astype(str).str.upper().str.strip()


nondrivers = pd.read_table(
    "data/processed/non_driver_genes.txt",
    # "../data/processed/5263_non_driver_genes.txt",
    header=None
)[0].astype(str).str.upper().str.strip()


drivers = set(drivers) & omics_genes
nondrivers = set(nondrivers) & omics_genes


print(
    f"Drivers after omics filtering: {len(drivers)}"
)

print(
    f"Non-drivers after omics filtering: {len(nondrivers)}"
)



# ==============================
# 2. Load enrichment
# ==============================

enrichment = pd.read_csv(
    "../reactome_embedding/results/enrichment/enrichment_simple.csv"
)[
    [
        "stId",
        "p_value",
        "significance"
    ]
]


enrichment_dict = (
    enrichment
    .set_index("stId")
    .to_dict("index")
)



# ==============================
# 3. Load pathway genes
# ==============================

pathway_genes_raw = pd.read_csv(
    "../data/processed/pathways_mapped_all_genes.tsv",
    sep="\t"
)


pathway_to_genes = {}

removed_gene_count = 0


for _, row in pathway_genes_raw.iterrows():

    pid = row["PathwayID"]


    genes = [

        str(g).upper().strip()

        for g in row[1:]

        if pd.notna(g)
    ]


    # --------------------------
    # IMPORTANT:
    # keep only omics genes
    # --------------------------

    before = len(genes)


    genes = [

        g for g in genes

        if g in omics_genes
    ]


    removed_gene_count += (
        before - len(genes)
    )


    if RANDOM_TRUNCATION:

        random.shuffle(genes)


    genes = genes[
        :MAX_GENES_PER_PATHWAY
    ]


    if genes:

        pathway_to_genes[pid] = genes



print(
    f"✅ Loaded pathways: {len(pathway_to_genes)}"
)

print(
    f"🧹 Removed non-omics genes: {removed_gene_count}"
)



# ==============================
# 4. Load pathway relations
# ==============================

pathway_relations = pd.read_csv(
    "../data/processed/ReactomePathwaysRelation_filtered.tsv",
    sep="\t",
    header=None,
    names=[
        "PathwayA",
        "PathwayB"
    ]
)


valid_pathways = set(
    enrichment["stId"]
)


pathway_relations = pathway_relations[
    pathway_relations["PathwayA"].isin(valid_pathways)
    &
    pathway_relations["PathwayB"].isin(valid_pathways)
]


print(
    f"✅ Filtered relations: {len(pathway_relations)}"
)



# ==============================
# 5. Generate gene pairs
# ==============================

gene_pairs = []


for _, row in tqdm(
    pathway_relations.iterrows(),
    total=len(pathway_relations)
):

    pA = row["PathwayA"]
    pB = row["PathwayB"]


    genesA = pathway_to_genes.get(
        pA,
        []
    )

    genesB = pathway_to_genes.get(
        pB,
        []
    )


    if not genesA or not genesB:
        continue



    enrich_info = enrichment_dict.get(
        pA
    )


    if enrich_info is None:
        continue


    pval = enrich_info["p_value"]

    sig = enrich_info["significance"]



    for gA, gB in product(
        genesA,
        genesB
    ):

        # remove self loops

        if gA == gB:
            continue


        gene_pairs.append(
            (
                pA,
                gA,
                pB,
                gB,
                pval,
                sig
            )
        )



print(
    f"✅ Raw pairs: {len(gene_pairs)}"
)



# ==============================
# 6. DataFrame
# ==============================

df = pd.DataFrame(
    gene_pairs,
    columns=[
        "PathwayA",
        "Gene1",
        "PathwayB",
        "Gene2",
        "pvalue",
        "significance"
    ]
)



# ==============================
# 7. Remove duplicates
# ==============================


df.drop_duplicates(
    inplace=True
)


df["pair_key"] = df.apply(

    lambda x:

    tuple(sorted(
        [
            (
                x["PathwayA"],
                x["Gene1"]
            ),

            (
                x["PathwayB"],
                x["Gene2"]
            )
        ]
    )),

    axis=1
)



df.drop_duplicates(
    subset="pair_key",
    inplace=True
)


df.drop(
    columns="pair_key",
    inplace=True
)



print(
    f"✅ Unique pairs: {len(df)}"
)



# ==============================
# 8. Gene type
# ==============================

def get_gene_type(row):

    return int(

        row["Gene1"] in drivers

        or

        row["Gene2"] in drivers
    )



df["gene_type"] = (
    df.progress_apply(
        get_gene_type,
        axis=1
    )
)



# ==============================
# 9. Limit genes/pathway
# ==============================

def limit_per_pathway(
    data,
    max_genes=100
):

    output=[]


    for pid, group in data.groupby(
        "PathwayA"
    ):


        driver_df = group[
            group["gene_type"]==1
        ]


        other_df = group[
            group["gene_type"]!=1
        ]


        need = max_genes - len(driver_df)



        if need > 0:

            sampled = other_df.sample(

                n=min(
                    need,
                    len(other_df)
                ),

                random_state=RANDOM_STATE
            )


            final = pd.concat(
                [
                    driver_df,
                    sampled
                ]
            )

        else:

            final = driver_df.sample(
                max_genes,
                random_state=RANDOM_STATE
            )



        # TP53 preservation

        if "TP53" in omics_genes:

            tp53 = group[
                group["Gene1"]=="TP53"
            ]


            final = pd.concat(
                [
                    final,
                    tp53
                ]
            ).drop_duplicates()



        output.append(
            final
        )


    return pd.concat(
        output,
        ignore_index=True
    )



df_final = limit_per_pathway(
    df,
    MAX_GENES_PER_PATHWAY
)



# ==============================
# 10. Final safety filter
# ==============================


df_final = df_final[
    df_final["Gene1"].isin(omics_genes)
    &
    df_final["Gene2"].isin(omics_genes)
]



print(
    f"✅ Final omics-compatible pairs: {len(df_final)}"
)



# ==============================
# 11. Save
# ==============================


output_path = (

    f"data/processed/"
    f"pathway_informed_gene_gene_pairs_omics_{MAX_GENES_PER_PATHWAY}.csv"

)



df_final.to_csv(
    output_path,
    index=False
)



print(
    f"✅ Saved → {output_path}"
)

✅ Gene universe from driver/non-driver labels: 17506 genes
Drivers after omics filtering: 763
Non-drivers after omics filtering: 16743


/var/folders/z_/4_txl3tn61g8nprq0_s7tsbc0000gn/T/ipykernel_1890/3605437270.py:106: DtypeWarning: Columns (340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,56

✅ Loaded pathways: 2359
🧹 Removed non-omics genes: 52702
✅ Filtered relations: 1900


  0%|          | 0/1900 [00:00<?, ?it/s]

✅ Raw pairs: 11201636
✅ Unique pairs: 11201636


  0%|          | 0/11201636 [00:00<?, ?it/s]

✅ Final omics-compatible pairs: 221679
✅ Saved → data/processed/pathway_informed_gene_gene_pairs_omics_500.csv
